In [1]:
import importlib as il
import numpy as np
import more_itertools as mit
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import scipy.linalg as spl
import linetimer as lt

import gurobipy as gp

import gurobi_utils as gu
import dikin_utils as du
import plot_utils as pu

import example_loader as el
import miplib_loader as ml
import jsplib_loader as jl
import knapsack_loader as kl

status_lookup = {getattr(gp.GRB.Status, k): k for k in gp.GRB.Status.__dir__() if "A" <= k[0] <= "Z"}

%matplotlib inline

env = gp.Env(empty=True)
env.setParam("OutputFlag", 0)
env.start()

<gurobipy.Env, Parameter changes: WLSAccessID=(user-defined), WLSSecret=(user-defined), LicenseID=2586148, OutputFlag=0>

In [ ]:
def get_A_b_c_l_u(mdl: gp.Model):
    mdl.update()
    A = mdl.getA()
    b = np.array(mdl.getAttr("RHS")).reshape(-1, 1)
    c = np.array(mdl.getAttr("Obj"))
    l = np.array(mdl.getAttr("LB"))
    u = np.array(mdl.getAttr("UB"))
    return A, b, c, l , u

def get_exact_skew_transform(xr, round=True):
    m = gp.Model()
    m.params.LogToConsole = 0
    x = m.addMVar(shape=(xr.shape[0], xr.shape[0]), lb=0, vtype=gp.GRB.CONTINUOUS)
    m.addConstr(x @ xr >= 1)
    m.addConstr(x.diagonal() == 1)
    m.setObjective(x.sum(), sense=gp.GRB.MINIMIZE)
    m.optimize()
    assert m.Status == gp.GRB.OPTIMAL, f"Status: {m.Status}"
    if round:
        return np.ceil(x.X) # rounding ruins the unimodularity
    return x.X

def get_skew_transform(xr, mul=1):
    # could solve the linear system: S*xr = xw, with Sii == 1,
    # and then round the values away from zero. 
    # alternate plan:
    # just try to use the previous value and hope it's not zero:
    xr = xr.flatten()
    assert np.all(xr >= 0)  # assumption for knapsack
    L = np.eye(xr.shape[0])
    U = np.eye(xr.shape[0])
    t = 1.4
    q = t / xr
    p = np.argmin(np.ceil(q) - q)
    assert q[p] > 0.0
    # v = np.ceil(q[p]) * mul
    # for i in range(xr.shape[0]):
    #     if i < p:
    #         U[i, p] = v
    #     elif i > p:
    #         L[i, p] = v
    #     elif i < xr.shape[0] - 1 and q[i+1] > 0: # not to the last row; check next col
    #         U[i, i+1] = np.ceil(q[i+1])
    #     elif i > 0 and q[i-1] > 0:
    #         L[i, i-1] = np.ceil(q[i-1])
    #     else:
    #         raise ValueError("Unexpected case encountered.")
    for i in range(xr.shape[0]):
        if i < xr.shape[0] - 1 and 0 < q[i+1] < np.inf: # not to the last row; check next col
            U[i, i+1] = np.ceil(q[i+1]) * mul
        elif i > 0 and 0 < q[i-1] < np.inf:
            L[i, i-1] = np.ceil(q[i-1]) * mul

    return L @ U

def construct_shear_matrix(x1, x2, tol=1e-10):
    """
    Constructs an n x n shear matrix S such that S @ x1 = x2 efficiently.
    
    Parameters:
    - x1: numpy array of shape (n,), nonnegative with at least one positive component.
    - x2: numpy array of shape (n,), nonnegative, x2 >= x1 component-wise.
    - tol: tolerance for numerical stability.
    
    Returns:
    - S: numpy array of shape (n, n), the shear matrix.
    """
    n = len(x1)
    
    # Validate inputs
    if not np.all(x2 >= x1) or not np.all(x2 >= 0):
        raise ValueError("x2 must be >= x1 and nonnegative")
    if np.all(x1 <= tol) and not np.all(x2 <= tol):
        raise ValueError("x1 must have a positive component if x2 does")
    if np.allclose(x1, x2, atol=tol):
        return np.eye(n)
    
    # Find pivot
    pivot_idx = np.where(x1 > tol)[0]
    if not pivot_idx.size:
        raise ValueError("x1 must have at least one positive component")
    k = pivot_idx[0]
    
    # Permute if pivot isn’t last
    perm = None
    if k != n - 1:
        perm = list(range(n))
        perm[k], perm[-1] = perm[-1], perm[k]
        x1_perm = x1[perm]
        x2_perm = x2[perm]
    else:
        x1_perm = x1
        x2_perm = x2
    
    # Construct S_upper
    S_upper = np.eye(n)
    for i in range(n - 1):
        if x2_perm[i] > x1_perm[i]:
            S_upper[i, n - 1] = (x2_perm[i] - x1_perm[i]) / x1_perm[n - 1]
    
    # Construct S_n
    S_n = np.eye(n)
    v = S_upper @ x1_perm  # Intermediate vector
    if x2_perm[n - 1] > x1_perm[n - 1]:
        if v[0] < tol:
            raise ValueError("First component of v is too small")
        S_n[n - 1, 0] = (x2_perm[n - 1] - x1_perm[n - 1]) / v[0]
    
    # Compute S in permuted space
    S_perm = S_n @ S_upper
    
    # Reverse permutation
    if perm is not None:
        P = np.eye(n)[perm]
        S = P.T @ S_perm @ P
    else:
        S = S_perm
    
    return S

# tx1 = np.array([0.2, 0.0, 0.6, 1.])
# tx1 = np.array([1.0, 0.0])
tx1 = np.random.random_sample(10)
txu = construct_shear_matrix(tx1, np.ones_like(tx1)*1.5)
txr = np.round(txu)
# tx2 = get_skew_transform(tx1, -1)
print("DET:", np.linalg.det(txu))
print("DET round:", np.linalg.det(txr))
# print(txu)
print(txr @ tx1)

DET: 1.0
DET round: 0.0
[1.21888887 1.52462042 1.35484963 1.4606352  1.53797022 1.68536915
 1.293871   1.46348498 1.68306075 1.21888887]


In [1]:
il.reload(kl)
il.reload(gu)

instances = kl.generate(3, 2, 2, 0, 1, 4, equality=False, seed=42)
for model in instances:
    model.params.LogToConsole = 0
    niv = model.NumIntVars  # keep before relaxation
    gu.relax_int_or_bin_to_continuous(model)
    model.optimize()
    x1 = np.array([v.X for v in model.getVars()])
    
    gu.standardize_gt_to_lt(model)
    A, b, c, lb, ub = get_A_b_c_l_u(model)
    A = A.todense()
    print(f"Model {model.ModelName}: min {c}x: {A}x <= {b})")

    senses = np.array(model.getAttr("Sense"))
    fig = pu.plot_constraints_lte(model.ModelName, A, b, lb, ub, senses, x_bounds=(-2, 4), y_bounds=(-2, 4))

    U = construct_shear_matrix(x1, np.ones_like(x1)*1.5)
    Uinv = np.linalg.inv(U)
    
    # Uinv = np.array([[1.0, 0.0], [0.4, 1.0]])
    # U = np.linalg.inv(Uinv)

    model2 = gu.apply_transform(model, Uinv, np.zeros_like(x1), ignore_bounds=False)
    senses = np.array(model2.getAttr("Sense"))
    A, b, c, lb, ub = get_A_b_c_l_u(model2)
    fig = pu.plot_constraints_lte(model2.ModelName + ' Skewed', A, b, lb, ub, senses, x_bounds=(-2, 4), y_bounds=(-2, 4))

    model2.params.LogToConsole = 0
    model2.optimize()
    x2 = np.array([v.X for v in model2.getVars()])
    added = 0
    for (rvar, lt) in gu.find_cuttable_rows(model2, range(niv)):
        added += 1
        if not lt:
            model2.addConstr(rvar >= np.ceil(rvar.X + 1e-5))
        else:
            model2.addConstr(rvar <= np.floor(rvar.X - 1e-5))

    model2.optimize()
    x3 = np.array([v.X for v in model2.getVars()])
    print("Added", added, "Result", x2, x3, Uinv @ x3)

NameError: name 'il' is not defined

In [20]:
il.reload(kl)
il.reload(gu)

instances = kl.generate(3, 2, 3, 0, 1, 5, equality=False, seed=42, env=env)
for model in instances:
    model.params.LogToConsole = 0
    niv = model.NumIntVars  # keep before relaxation
    model.optimize()
    x0 = np.array([v.X for v in model.getVars()])

    gu.relax_int_or_bin_to_continuous(model)
    model2 = model

    for _ in range(10):
        model2.optimize()
        x1 = np.array([v.X for v in model2.getVars()])
        
        # U = construct_shear_matrix(x1, np.ones_like(x1)*1.5)
        # Uinv = np.linalg.inv(U)
        fractional_indices = x1 != np.floor(x1)
        if not np.any(fractional_indices):
            print("   All variables are integral!", model2.ModelName, x0, x1, Uinv @ x1)
            break
        first_fractional_index = np.argmax(fractional_indices)
        U = np.eye(x1.shape[0])
        U[:, first_fractional_index] = 1
        Uinv = np.linalg.inv(U)

        model2 = gu.apply_transform(model2, Uinv, np.zeros_like(x1), env=env)
        model2.params.LogToConsole = 0
        model2.optimize()
        x2 = np.array([v.X for v in model2.getVars()])
        added = 0
        for (rvar, lt) in gu.find_cuttable_rows(model2, range(niv)):
            added += 1
            if not lt:
                rvar.LB = np.ceil(rvar.X + 1e-5)
                # model2.addConstr(rvar >= np.ceil(rvar.X + 1e-5))
            else:
                rvar.UB = np.floor(rvar.X - 1e-5)
                # model2.addConstr(rvar <= np.floor(rvar.X - 1e-5))
        if added == 0:
            print("   No cuts added! Expecting optimum at 0.", model2.ModelName, x0, x1, U @ x1, x2)
            break
        print(f"   Added {added} cuts", x1, x2)

   Relaxed 3 variables on knapsack_2_3_0
   Added 1 cuts [0.75 0.   1.  ] [0.75 0.75 1.75]
   No cuts added! Expecting optimum at 0. knapsack_2_3_0_transformed_transformed [1. 0. 0.] [0.  0.6 1. ] [0.6 0.6 1.6] [0.6 0.6 1.6]
   Relaxed 3 variables on knapsack_2_3_1
   Added 1 cuts [0.  0.9 0.8] [0.9 0.9 1.7]
   No cuts added! Expecting optimum at 0. knapsack_2_3_1_transformed_transformed [0. 0. 1.] [0.2 0.  1. ] [0.2 0.2 1.2] [0.2 0.2 1.2]
   Relaxed 3 variables on knapsack_2_3_2
   No cuts added! Expecting optimum at 0. knapsack_2_3_2_transformed [0. 0. 1.] [0.75   0.0625 1.    ] [0.75   0.8125 1.75  ] [0.75   0.8125 1.75  ]
